# Exploratory Data Analysis — Pearls AQI Predictor

This notebook is a thin, interactive front end over `src/eda.py`. The analysis
functions live in the package rather than in notebook cells so that the
notebook, the generated `reports/eda_report.md` and the Streamlit EDA tab all
compute the same numbers and cannot drift apart.

**Prerequisite:** run the backfill first so the Feature Store has data.

```bash
python -m src.pipelines.backfill --days 365
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import config, eda

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

config.summary()

## 1. Load the engineered features

In [ ]:
df = eda.load()
print(f"{len(df):,} hourly rows x {len(df.columns)} columns")
print(f"{df['ts'].min()}  ->  {df['ts'].max()}")
df[["ts", "aqi", "pm2_5", "pm10", "temperature", "wind_speed"]].tail()

## 2. Headline statistics

`describe()` returns the same dictionary the written report and the dashboard quote.

In [ ]:
stats = eda.describe(df)
for k, v in stats.items():
    print(f"{k:28} {v}")

## 3. The series over time

Shaded bands are the EPA AQI categories; the dashed line is the alert threshold.

In [ ]:
eda.plot_timeseries(df)
plt.show()

## 4. Distribution

The distribution is right-skewed: typical conditions cluster near the median,
but the tail reaches far higher. That skew is why the neural networks are
trained with a Huber loss — the rare severe episodes are exactly what a
forecast needs to get right, and a squared loss lets them dominate the gradient.

In [ ]:
eda.plot_distribution(df)
plt.show()

## 5. Seasonality

Daily, weekly and monthly cycles. The daily cycle is why hour-of-day is encoded
cyclically (sin/cos) rather than as a raw 0–23 integer — otherwise 23:00 and
00:00 sit at opposite ends of the number line despite being adjacent in time.

In [ ]:
eda.plot_seasonality(df)
plt.show()

## 6. Autocorrelation — justifying the lag window

This is the single most important plot for the modelling decisions. It shows
how far back AQI carries information, and therefore how far forward it is
reasonable to forecast.

In [ ]:
ac = eda.autocorrelation(df, max_lag=168)
print(ac.loc[[1, 3, 6, 12, 24, 48, 72, 120, 168]].round(3))
eda.plot_autocorrelation(df)
plt.show()

## 7. Meteorological drivers

Wind speed is the dominant control: stagnant air traps particulates near the
surface. This motivates the `stagnation` = 1/(1 + wind_speed) feature and the
decomposition of wind direction into u/v vector components.

In [ ]:
print(eda.correlations(df)["aqi"].drop("aqi").sort_values(key=abs, ascending=False).round(3))
eda.plot_weather_relationships(df)
plt.show()

In [ ]:
eda.plot_correlation(df)
plt.show()

## 8. Which engineered features track AQI most strongly?

A first look at what the models are likely to lean on — cross-check this
against the SHAP ranking after training, since correlation and actual model
reliance are not the same thing.

In [ ]:
print(eda.target_correlations(df, top_n=20).round(3))
eda.plot_target_correlations(df)
plt.show()

## 9. Regenerate the written report

Writes `reports/eda_report.md` and all figures to `reports/figures/`.

In [ ]:
result = eda.run()
print(f"{len(result['figures'])} figures written")
print((config.REPORT_DIR / "eda_report.md").read_text(encoding="utf-8")[:2000])

## 10. Conclusions carried into modelling

1. **Lag features carry most of the signal**, so a persistence baseline is
   genuinely hard to beat at +24 h. Every candidate is scored against it and the
   training pipeline flags any winner that fails to clear it.
2. **The train/test split must be chronological.** With lag features this dense,
   a random split leaks the future and inflates R².
3. **Direct multi-horizon beats recursive.** Separate heads for +24/48/72 h avoid
   compounding one-step error across three days.
4. **Skew argues for robust losses and tree ensembles**, which is what the model
   comparison generally bears out.

Next: `python -m src.pipelines.training_pipeline`